# Fine-Tuning Laya on `LocalLLaMA/typed-decisions` & Evaluation (Single T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NandhaKishorM/laya/blob/main/notebooks/laya_finetune_typed_decisions_colab.ipynb)
[![PyPI version](https://img.shields.io/pypi/v/laya.svg)](https://pypi.org/project/laya/)
[![Hugging Face Model](https://img.shields.io/badge/%F0%9F%A4%97%20Model-convaiinnovations%2Flaya-blue)](https://huggingface.co/convaiinnovations/laya)
[![Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20Dataset-LocalLLaMA%2Ftyped--decisions-green)](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)

This notebook fine-tunes **Laya** (`convaiinnovations/laya`) on the **1,200 training cases (6,000 typed decisions)** of the [LocalLLaMA/typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions) benchmark on a **single free T4 GPU (16 GB)** using **RLCD (Reinforcement Learning for Calibrated Decisions)**.

After fine-tuning (~10-15 minutes), it evaluates the model against the **400 test cases (2,000 decisions)** and compares head-to-head with **TypeSafe Jev 1.13.0**, then lets you push the new checkpoint to Hugging Face.

### The 4 Production Workflows:
1. `agent_trace_observability` (300 train / 100 test)
2. `customer_service` (300 train / 100 test)
3. `invoice_processing` (300 train / 100 test)
4. `security_incidents` (300 train / 100 test)


## 1. Environment & GPU Check
Verify the T4 GPU is active (`Runtime -> Change runtime type -> T4 GPU`).


In [ ]:
!nvidia-smi
import os, torch
assert torch.cuda.is_available(), "No GPU detected! Set Runtime -> Change runtime type -> T4 GPU"
props = torch.cuda.get_device_properties(0)
print(f"Connected to: {props.name} | Total VRAM: {props.total_memory / 1e9:.1f} GB")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


## 2. Install Dependencies


In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate
import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)


## 3. Load Base Model & Tokenizer
We load `convaiinnovations/laya` (421M parameters) in FP16 with gradient checkpointing enabled for single-GPU training.


In [ ]:
import os, json, torch
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_model, QTYPES, render_options, build_sequence, proper_reward, confidence_from_probs
from transformers import AutoTokenizer
from safetensors.torch import load_file

MODEL_ID = "convaiinnovations/laya"
print(f"Downloading base model weights from {MODEL_ID}...")
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)

with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

# Optimize config for single T4 fine-tuning
cfg["gradient_checkpointing"] = True
cfg["max_tokens_per_batch"] = 4096
cfg["head_layers"] = 2
cfg["max_len"] = 512
cfg["head_max_len"] = 192

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))

# Load fine-tuned weights
weights = load_file(os.path.join(model_dir, "model.safetensors"))
model.load_state_dict(weights, strict=True)

device = torch.device("cuda")
model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.head_checkpointing = True
model.to(device)
model.train()

print(f"Successfully loaded Laya ({sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params) on {device}!")


## 4. Download & Preprocess `typed-decisions` (Train Split)
We download all 1,200 training cases and format them with soft gold targets from the teacher.


In [ ]:
from datasets import load_dataset
import pandas as pd
import json

print("Downloading LocalLLaMA/typed-decisions (split: train)...")
ds_train = load_dataset("LocalLLaMA/typed-decisions", "all", split="train")
print(f"Loaded {len(ds_train)} training workflow cases.")

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    
    # Normalize soft target
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {
        "ids": seq,
        "markers": markers,
        "qtype": QTYPES[t],
        "target": target,
        "label": label
    }

training_items = []
print("Preprocessing training records into tokenized items...")

for row in ds_train:
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    
    for qid, q in questions.items():
        if qid in gold:
            it = build_training_item(state, q, gold[qid])
            if it:
                training_items.append(it)

print(f"Preprocessed {len(training_items)} training sequences across 1,200 cases.")


## 5. RLCD Policy Gradient Fine-Tuning Loop
We train the model using **strictly proper scoring rules** (Log + Spherical + Ranked Probability Score) with a GRPO-style group baseline ($G=4$ samples per question).


In [ ]:
import random, time
import numpy as np

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

EPOCHS = 2
BATCH_SIZE = 16      # 16 cases * 5 questions = 80 sequences per batch
GROUP_SIZE = 4       # Number of noisy candidate logits sampled per question (GRPO baseline)
LR_ENCODER = 1.5e-5  # Encoder fine-tuning rate
LR_HEAD = 1.5e-4     # Head fine-tuning rate
SIGMA_START = 0.5    # Starting exploration noise
SIGMA_END = 0.2      # Final exploration noise

enc_params = [p for n, p in model.named_parameters() if n.startswith("encoder.")]
head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]

optimizer = torch.optim.AdamW([
    {"params": enc_params, "lr": LR_ENCODER},
    {"params": head_params, "lr": LR_HEAD}
], weight_decay=0.01)

scaler = torch.amp.GradScaler("cuda", enabled=True)

print("Starting RLCD fine-tuning loop...")
t0_train = time.time()

for epoch in range(EPOCHS):
    random.shuffle(training_items)
    epoch_loss, epoch_rew, n_batches = 0.0, 0.0, 0
    progress = epoch / max(1, EPOCHS - 1)
    sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
    
    # Process in micro-batches
    for b_idx in range(0, len(training_items), BATCH_SIZE * 5):
        chunk = training_items[b_idx:b_idx + BATCH_SIZE * 5]
        if not chunk:
            continue
        
        batch = collate_train_batch(chunk, tok.pad_token_id)
        optimizer.zero_grad()
        
        with torch.autocast("cuda", dtype=torch.float16):
            logits, act = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["marker_pos"].to(device),
                batch["marker_mask"].to(device),
                batch["qtype"].to(device)
            )
        
        logits = logits.float()
        mask = batch["marker_mask"].to(device)
        k = mask.sum(-1, keepdim=True).float()
        
        # 1. Sample G noisy logit distributions with zero-mean projection
        eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
        eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
        z = logits.detach().unsqueeze(0) + eps
        q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
        
        # 2. Evaluate proper scoring reward (Log + Spherical + RPS)
        target = batch["target"].to(device)
        with torch.no_grad():
            r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask)
            # 3. GRPO-style Group Mean Baseline Advantage
            adv = r - r.mean(0, keepdim=True)
            adv = adv / (adv.std() + 1e-6)
        
        # 4. Policy gradient loss
        logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
        loss = -(adv * logp).mean()
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        epoch_rew += r.mean().item()
        n_batches += 1
        
        if (n_batches % 10) == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {n_batches} | Loss: {loss.item():.4f} | Reward: {r.mean().item():.3f}")

    avg_loss = epoch_loss / max(1, n_batches)
    avg_rew = epoch_rew / max(1, n_batches)
    print(f"=== Epoch {epoch + 1}/{EPOCHS} Finished in {time.time() - t0_train:.1f}s | Avg Loss: {avg_loss:.4f} | Avg Reward: {avg_rew:.3f} ===")

print(f"\nFine-tuning completed in {time.time() - t0_train:.1f}s!")


## 6. Save the Fine-Tuned Checkpoint


In [ ]:
OUTPUT_DIR = "/content/laya_finetuned_typed_decisions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Save fp16 weights
sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
weights_file = os.path.join(OUTPUT_DIR, "model.safetensors")
save_file(sd, weights_file)

# 2. Save encoder config and tokenizer
model.encoder.config.save_pretrained(os.path.join(OUTPUT_DIR, "encoder"))
tok.save_pretrained(os.path.join(OUTPUT_DIR, "tokenizer"))

# 3. Save config with fitted calibration parameters
cfg["fine_tuned"] = True
cfg["model_name"] = "laya-typed-decisions"
with open(os.path.join(OUTPUT_DIR, "rl_agent_config.json"), "w") as f:
    json.dump(cfg, f, indent=2)

print(f"Model successfully saved to {OUTPUT_DIR} (Total size: {os.path.getsize(weights_file) / 1e6:.1f} MB)!")


## 7. Evaluate on Benchmark (`test` split: 400 cases / 2,000 decisions)
We run inference on the official `test` split and calculate the exact benchmark metrics.


In [ ]:
print("Loading test split for evaluation...")
ds_test = load_dataset("LocalLLaMA/typed-decisions", "all", split="test")

# Load fine-tuned model with Laya runtime
agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")

predictions = []
latencies_ms = []

print(f"Evaluating {len(ds_test)} test cases...")
t0_eval = time.time()

for i, row in enumerate(ds_test):
    case_id = row["id"]
    workflow = row["workflow"]
    state = json.loads(row["state"])
    questions = json.loads(row["questions"])
    gold = json.loads(row["gold"])
    
    t0 = time.perf_counter()
    res = agent_ft.predict(state, questions)
    dt_ms = (time.perf_counter() - t0) * 1000
    latencies_ms.append(dt_ms)
    
    predictions.append({
        "id": case_id,
        "workflow": workflow,
        "pred": res["answers"],
        "gold": gold,
        "questions": questions,
        "latency_ms": dt_ms
    })
    
    if (i + 1) % 50 == 0 or (i + 1) == len(ds_test):
        print(f"  Progress: {i + 1}/{len(ds_test)} cases | Avg Latency: {np.mean(latencies_ms):.1f} ms/case")

print(f"Evaluated all {len(ds_test)} cases in {time.time() - t0_eval:.1f}s!")


## 8. Compute Official Benchmark Metrics & Compare Head-to-Head


In [ ]:
from laya.common import ece_score

accuracies = []
soft_accuracies = []
brier_scores = []
kl_divs = []
tv_distances = []
score_maes = []
within_one = []
all_confs = []
all_corrects = []

for item in predictions:
    pred_answers = item["pred"]
    gold_answers = item["gold"]
    questions = item["questions"]
    
    for qid, qdef in questions.items():
        p_ans = pred_answers[qid]
        g_ans = gold_answers[qid]
        q_type = qdef["type"]
        
        # 1. Choice
        if q_type == "choice":
            keys = list(qdef["criteria"].keys())
            pred_choice = p_ans["choice"]
            gold_label = g_ans["label"]
            
            is_corr = float(pred_choice == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(p_ans["confidence"])
            
            p_probs = np.array([p_ans["probabilities"].get(k, 1e-6) for k in keys])
            g_probs = np.array([g_ans["probabilities"].get(k, 1e-6) for k in keys])
            p_probs /= p_probs.sum()
            g_probs /= g_probs.sum()
            
            soft_accuracies.append(float((p_probs * g_probs).sum()))
            brier_scores.append(float(((p_probs - g_probs) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_probs - g_probs).sum()))
            kl_divs.append(float((g_probs * np.log(np.clip(g_probs / p_probs, 1e-12, 1e4))).sum()))
            
        # 2. Noul
        elif q_type == "noul":
            p_val = p_ans["noul"]
            g_val = g_ans.get("noul", g_ans.get("probabilities", {}).get("true", 0.5))
            gold_label = str(g_ans["label"]).lower()
            
            pred_label = "true" if p_val >= 0.5 else "false"
            is_corr = float(pred_label == gold_label)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(max(p_val, 1.0 - p_val))
            
            p_dist = np.array([1.0 - p_val, p_val])
            g_dist = np.array([1.0 - g_val, g_val])
            
            soft_accuracies.append(float((p_dist * g_dist).sum()))
            brier_scores.append(float(((p_dist - g_dist) ** 2).sum()))
            tv_distances.append(float(0.5 * np.abs(p_dist - g_dist).sum()))
            kl_divs.append(float((g_dist * np.log(np.clip(g_dist / p_dist, 1e-12, 1e4))).sum()))
            
        # 3. Score
        elif q_type == "score":
            p_score = p_ans["score"]
            g_score = g_ans.get("score", 0.0)
            score_maes.append(abs(p_score - g_score))
            
            # Within 1 level metric
            within_one.append(float(abs(p_score - g_score) <= 1.0))
            
            p_lvl = int(round(p_score))
            g_lvl = int(g_ans.get("label", int(round(g_score))))
            is_corr = float(p_lvl == g_lvl)
            accuracies.append(is_corr)
            all_corrects.append(is_corr)
            all_confs.append(p_ans.get("confidence", 0.5))

laya_acc = np.mean(accuracies)
laya_soft_acc = np.mean(soft_accuracies)
laya_brier = np.mean(brier_scores)
laya_kl = np.mean(kl_divs)
laya_tv = np.mean(tv_distances)
laya_ece = ece_score(np.array(all_confs), np.array(all_corrects))
laya_mae = np.mean(score_maes) if score_maes else 0.0
laya_within1 = np.mean(within_one) if within_one else 0.0
laya_latency = np.percentile(latencies_ms, 50)

comparison_data = [
    {
        "Model": "TypeSafe Jev 1.13.0",
        "Kind": "general",
        "Accuracy": 0.727,
        "Soft Acc": 0.580,
        "Brier": 0.148,
        "ECE": 0.144,
        "Score MAE": 0.391,
        "Within 1 Level": "0.952",
        "ms/case": 710,
        "Cost/Case": "$0.0004 (API)"
    },
    {
        "Model": "Laya (Fine-Tuned)",
        "Kind": "fine-tuned",
        "Accuracy": round(laya_acc, 3),
        "Soft Acc": round(laya_soft_acc, 3),
        "Brier": round(laya_brier, 3),
        "ECE": round(laya_ece, 3),
        "Score MAE": round(laya_mae, 3),
        "Within 1 Level": f"{laya_within1:.3f}",
        "ms/case": round(laya_latency, 1),
        "Cost/Case": "$0.00 (Self-Hosted)"
    },
    {
        "Model": "ModernBERT-base (149M)",
        "Kind": "specialist",
        "Accuracy": 0.646,
        "Soft Acc": 0.542,
        "Brier": 0.119,
        "ECE": 0.179,
        "Score MAE": 0.444,
        "Within 1 Level": "0.931",
        "ms/case": 349,
        "Cost/Case": "$0.00"
    },
    {
        "Model": "Teacher Self-Agreement",
        "Kind": "ceiling",
        "Accuracy": 0.735,
        "Soft Acc": "-",
        "Brier": "-",
        "ECE": "-",
        "Score MAE": "-",
        "Within 1 Level": "-",
        "ms/case": "-",
        "Cost/Case": "-"
    }
]

df_comp = pd.DataFrame(comparison_data)
print("=== HEAD-TO-HEAD BENCHMARK TABLE ===\n")
print(df_comp.to_markdown(index=False))


## 9. (Optional) Push Fine-Tuned Model to Hugging Face Hub
Publish your newly fine-tuned checkpoint directly to Hugging Face.


In [ ]:
# Set your write token and repo name:
# from huggingface_hub import HfApi

# HF_TOKEN = "hf_YOUR_WRITE_TOKEN"
# NEW_REPO = "convaiinnovations/laya-typed-decisions"

# api = HfApi(token=HF_TOKEN)
# api.create_repo(NEW_REPO, repo_type="model", private=False, exist_ok=True)
# api.upload_folder(
#     folder_path=OUTPUT_DIR,
#     repo_id=NEW_REPO,
#     repo_type="model",
#     commit_message=f"Fine-tuned Laya on typed-decisions (Accuracy: {laya_acc:.3f})"
# )
# print(f"Model pushed successfully to https://huggingface.co/{NEW_REPO}!")
